# Credit Train — Feature Engineering

**Upstream:** `01_credit_train_eda.ipynb`
**Input:** `Datasets/credit_train.csv`
**Output:** Engineered feature matrix under `features/`

---

### Purpose

Transform the cleaned raw loan file into a modelling-ready feature set. The notebook encodes domain knowledge as **ratios, flags, bins, interactions, and missingness indicators**, then persists a documented matrix for selection and training.

### Engineering themes

1. Data cleaning aligned with the EDA contract
2. Ordinal and grouped categorical encodings
3. Affordability and leverage ratios (DTI, LTI, utilisation)
4. Risk flags (term, tenure, public records, purpose)
5. Score bands, quantile bins, and selected interactions
6. Feature dictionary and parquet/CSV export


## 0. Setup

Import libraries and define input / output paths for the engineered feature matrix.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.figsize"] = (10, 5)

ROOT = Path("../..")
DATA = ROOT / "Datasets" / "credit_train.csv"
OUT_DIR = Path("features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA.exists(), DATA.resolve()
print("Ready. Output dir:", OUT_DIR.resolve())


## 1. Load, deduplicate and quality fixes

Apply the EDA cleaning contract: drop empty rows, deduplicate `Loan ID`, repair credit-score scaling, replace sentinel loan amounts, and normalise category labels.


In [ ]:
raw = pd.read_csv(DATA)
print(f"Raw: {raw.shape[0]:,} x {raw.shape[1]}")

# Deduplicate on Loan ID (keep first)
df = raw.drop_duplicates(subset=["Loan ID"]).copy()
print(f"After Loan-ID dedup: {df.shape[0]:,}")

id_cols = ["Loan ID", "Customer ID"]
ids = df[id_cols].copy()

# Credit Score: values > 850 are score x 10
n_fix = (df["Credit Score"] > 850).sum()
df.loc[df["Credit Score"] > 850, "Credit Score"] = (
    df.loc[df["Credit Score"] > 850, "Credit Score"] / 10
)
print(f"Credit scores fixed: {n_fix:,}")

# Loan amount sentinel
n_sent = (df["Current Loan Amount"] == 99999999).sum()
df["Current Loan Amount"] = df["Current Loan Amount"].replace(99999999, np.nan)
print(f"Loan amount sentinels -> NaN: {n_sent:,}")

# Category normalisation
df["Home Ownership"] = df["Home Ownership"].replace({"HaveMortgage": "Home Mortgage"})
df["Purpose"] = df["Purpose"].str.lower().str.strip().str.replace(" ", "_")

# Target
df["Default"] = (df["Loan Status"] == "Charged Off").astype(int)
print(f"Default rate: {df['Default'].mean()*100:.2f}%")


## 2. Feature engineering functions

Reusable transforms for ordinal job tenure, purpose groups, affordability ratios, risk flags, score bands, and interaction terms.


In [ ]:
FEATURE_DICT = []

def register(name, family, description, dtype="float", source="engineered"):
    FEATURE_DICT.append({
        "feature": name,
        "family": family,
        "description": description,
        "dtype": dtype,
        "source": source,
    })

JOB_MAP = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
    "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9,
    "10+ years": 10, "n/a": np.nan,
}

def add_job_tenure(d):
    d = d.copy()
    d["job_years"] = d["Years in current job"].map(JOB_MAP)
    register("job_years", "ordinal", "Years in current job mapped to 0-10", "float", "Years in current job")
    d["job_years_missing"] = d["job_years"].isna().astype(int)
    register("job_years_missing", "missing_flag", "1 if job tenure missing", "int", "Years in current job")
    d["is_short_tenure"] = (d["job_years"].fillna(99) <= 1).astype(int)
    register("is_short_tenure", "binary_flag", "1 if job tenure <= 1 year", "int", "job_years")
    return d

def add_missing_flags(d):
    d = d.copy()
    for col, alias in [
        ("Credit Score", "credit_score"),
        ("Annual Income", "annual_income"),
        ("Current Loan Amount", "loan_amount"),
        ("Months since last delinquent", "months_delinquent"),
        ("Bankruptcies", "bankruptcies"),
        ("Maximum Open Credit", "max_open_credit"),
    ]:
        if col not in d.columns:
            continue
        flag = f"{alias}_missing"
        d[flag] = d[col].isna().astype(int)
        register(flag, "missing_flag", f"1 if {col} is missing", "int", col)
    return d

def add_ratios(d):
    d = d.copy()
    d["dti"] = np.where(
        d["Annual Income"] > 0,
        (d["Monthly Debt"] * 12) / d["Annual Income"],
        np.nan,
    )
    if d["dti"].notna().any():
        d["dti"] = d["dti"].clip(upper=d["dti"].quantile(0.995))
    register("dti", "ratio", "(Monthly Debt x 12) / Annual Income, clipped at 99.5th pct", "float",
             "Monthly Debt, Annual Income")

    d["lti"] = np.where(
        d["Annual Income"] > 0,
        d["Current Loan Amount"] / d["Annual Income"],
        np.nan,
    )
    if d["lti"].notna().any():
        d["lti"] = d["lti"].clip(upper=d["lti"].quantile(0.995))
    register("lti", "ratio", "Current Loan Amount / Annual Income, clipped at 99.5th pct", "float",
             "Current Loan Amount, Annual Income")

    d["credit_util"] = np.where(
        d["Maximum Open Credit"] > 0,
        d["Current Credit Balance"] / d["Maximum Open Credit"],
        np.nan,
    )
    d["credit_util"] = d["credit_util"].clip(upper=2.0)
    register("credit_util", "ratio", "Current Credit Balance / Max Open Credit, capped at 2.0", "float",
             "Current Credit Balance, Maximum Open Credit")

    d["debt_to_loan"] = np.where(
        d["Current Loan Amount"] > 0,
        d["Monthly Debt"] / d["Current Loan Amount"],
        np.nan,
    )
    if d["debt_to_loan"].notna().any():
        d["debt_to_loan"] = d["debt_to_loan"].clip(upper=d["debt_to_loan"].quantile(0.995))
    register("debt_to_loan", "ratio", "Monthly Debt / Current Loan Amount", "float",
             "Monthly Debt, Current Loan Amount")

    d["balance_to_income"] = np.where(
        d["Annual Income"] > 0,
        d["Current Credit Balance"] / d["Annual Income"],
        np.nan,
    )
    if d["balance_to_income"].notna().any():
        d["balance_to_income"] = d["balance_to_income"].clip(upper=d["balance_to_income"].quantile(0.995))
    register("balance_to_income", "ratio", "Current Credit Balance / Annual Income", "float",
             "Current Credit Balance, Annual Income")

    d["credit_headroom"] = d["Maximum Open Credit"] - d["Current Credit Balance"]
    register("credit_headroom", "derived", "Max Open Credit - Current Credit Balance", "float",
             "Maximum Open Credit, Current Credit Balance")
    return d

def add_credit_flags(d):
    d = d.copy()
    d["has_delinquency_hist"] = d["Months since last delinquent"].notna().astype(int)
    register("has_delinquency_hist", "binary_flag",
             "1 if Months since last delinquent is present", "int", "Months since last delinquent")

    d["has_credit_problems"] = (d["Number of Credit Problems"].fillna(0) > 0).astype(int)
    register("has_credit_problems", "binary_flag", "1 if Number of Credit Problems > 0", "int",
             "Number of Credit Problems")

    d["has_bankruptcy"] = (d["Bankruptcies"].fillna(0) > 0).astype(int)
    register("has_bankruptcy", "binary_flag", "1 if Bankruptcies > 0", "int", "Bankruptcies")

    d["has_tax_lien"] = (d["Tax Liens"].fillna(0) > 0).astype(int)
    register("has_tax_lien", "binary_flag", "1 if Tax Liens > 0", "int", "Tax Liens")

    d["has_any_public_record"] = (
        (d["has_credit_problems"] + d["has_bankruptcy"] + d["has_tax_lien"]) > 0
    ).astype(int)
    register("has_any_public_record", "binary_flag",
             "1 if any of credit problems / bankruptcy / tax lien", "int",
             "has_credit_problems, has_bankruptcy, has_tax_lien")

    d["credit_problem_count"] = (
        d["Number of Credit Problems"].fillna(0)
        + d["Bankruptcies"].fillna(0)
        + d["Tax Liens"].fillna(0)
    )
    register("credit_problem_count", "count", "Sum of credit problems + bankruptcies + tax liens",
             "float", "Number of Credit Problems, Bankruptcies, Tax Liens")
    return d

def add_simple_categoricals(d):
    d = d.copy()
    d["is_long_term"] = (d["Term"] == "Long Term").astype(int)
    register("is_long_term", "binary_flag", "1 if Term = Long Term", "int", "Term")

    d["is_renter"] = (d["Home Ownership"] == "Rent").astype(int)
    register("is_renter", "binary_flag", "1 if Home Ownership = Rent", "int", "Home Ownership")

    d["is_mortgage"] = (d["Home Ownership"] == "Home Mortgage").astype(int)
    register("is_mortgage", "binary_flag", "1 if Home Ownership = Home Mortgage", "int", "Home Ownership")

    d["is_homeowner"] = (d["Home Ownership"] == "Own Home").astype(int)
    register("is_homeowner", "binary_flag", "1 if Home Ownership = Own Home", "int", "Home Ownership")
    return d

PURPOSE_GROUPS = {
    "debt_consolidation": "debt_consolidation",
    "home_improvements": "home",
    "buy_house": "home",
    "buy_a_car": "vehicle",
    "business_loan": "business",
    "small_business": "business",
    "medical_bills": "medical",
    "take_a_trip": "discretionary",
    "vacation": "discretionary",
    "wedding": "discretionary",
    "major_purchase": "discretionary",
    "moving": "discretionary",
    "educational_expenses": "education",
    "renewable_energy": "other",
    "other": "other",
}

def add_purpose_group(d):
    d = d.copy()
    d["purpose_group"] = d["Purpose"].map(PURPOSE_GROUPS).fillna("other")
    register("purpose_group", "categorical", "Purpose collapsed into stable groups", "str", "Purpose")
    for grp in ["debt_consolidation", "home", "business", "discretionary", "medical", "vehicle", "other"]:
        col = f"purpose_{grp}"
        d[col] = (d["purpose_group"] == grp).astype(int)
        register(col, "binary_flag", f"1 if purpose_group = {grp}", "int", "purpose_group")
    return d

def add_bins(d):
    d = d.copy()

    def qbin(series, q=5):
        try:
            binned = pd.qcut(series, q=q, duplicates="drop")
            return binned.cat.codes.replace(-1, np.nan)
        except ValueError:
            return pd.Series(np.nan, index=series.index)

    d["credit_score_bin"] = qbin(d["Credit Score"])
    register("credit_score_bin", "bin", "Credit Score quintile code (0-4)", "float", "Credit Score")

    d["income_bin"] = qbin(d["Annual Income"])
    register("income_bin", "bin", "Annual Income quintile code (0-4)", "float", "Annual Income")

    d["loan_amount_bin"] = qbin(d["Current Loan Amount"])
    register("loan_amount_bin", "bin", "Loan Amount quintile code (0-4)", "float", "Current Loan Amount")

    d["dti_bin"] = qbin(d["dti"])
    register("dti_bin", "bin", "DTI quintile code (0-4)", "float", "dti")

    d["util_bin"] = qbin(d["credit_util"])
    register("util_bin", "bin", "Credit utilisation quintile code (0-4)", "float", "credit_util")

    def score_band(s):
        if pd.isna(s):
            return "missing"
        if s < 650:
            return "poor"
        if s < 700:
            return "fair"
        if s < 740:
            return "good"
        return "excellent"

    d["credit_score_band"] = d["Credit Score"].apply(score_band)
    register("credit_score_band", "categorical", "Domain credit score band", "str", "Credit Score")
    for band in ["poor", "fair", "good", "excellent", "missing"]:
        col = f"score_band_{band}"
        d[col] = (d["credit_score_band"] == band).astype(int)
        register(col, "binary_flag", f"1 if credit_score_band = {band}", "int", "credit_score_band")
    return d

def add_interactions(d):
    d = d.copy()
    d["longterm_x_renter"] = d["is_long_term"] * d["is_renter"]
    register("longterm_x_renter", "interaction", "Long-term loan AND renter", "int",
             "is_long_term, is_renter")

    d["longterm_x_lowscore"] = d["is_long_term"] * d.get("score_band_poor", 0)
    register("longterm_x_lowscore", "interaction", "Long-term loan AND poor credit band", "int",
             "is_long_term, score_band_poor")

    med_dti = d["dti"].median()
    d["high_dti_x_renter"] = ((d["dti"] > med_dti) & (d["is_renter"] == 1)).astype(int)
    register("high_dti_x_renter", "interaction", "Above-median DTI AND renter", "int", "dti, is_renter")

    d["short_tenure_x_longterm"] = d["is_short_tenure"] * d["is_long_term"]
    register("short_tenure_x_longterm", "interaction", "Short job tenure AND long-term loan", "int",
             "is_short_tenure, is_long_term")
    return d

def add_log_transforms(d):
    d = d.copy()
    for col, alias in [
        ("Annual Income", "log_annual_income"),
        ("Current Loan Amount", "log_loan_amount"),
        ("Monthly Debt", "log_monthly_debt"),
        ("Current Credit Balance", "log_credit_balance"),
        ("Maximum Open Credit", "log_max_open_credit"),
    ]:
        if col not in d.columns:
            continue
        d[alias] = np.log1p(d[col].clip(lower=0))
        register(alias, "transform", f"log1p({col})", "float", col)
    return d

print("Feature functions defined.")


## 3. Apply the full feature pipeline

Run every engineering step on the cleaned frame and inspect column growth.


In [ ]:
feat = df.copy()

feat = add_job_tenure(feat)
feat = add_missing_flags(feat)
feat = add_ratios(feat)
feat = add_credit_flags(feat)
feat = add_simple_categoricals(feat)
feat = add_purpose_group(feat)
feat = add_bins(feat)
feat = add_interactions(feat)
feat = add_log_transforms(feat)

print(f"Frame after engineering: {feat.shape[0]:,} x {feat.shape[1]}")
print(f"Registered features: {len(FEATURE_DICT)}")


In [ ]:
feat_dict = pd.DataFrame(FEATURE_DICT).drop_duplicates(subset=["feature"])
print(feat_dict.groupby("family").size().sort_values(ascending=False).to_string())
print()
feat_dict.head(20)


## 4. Assemble the modelling feature set

Select numeric and binary columns suitable for scikit-learn, attach the target, and drop identifiers from the model matrix.


In [ ]:
RAW_NUMERIC = [
    "Current Loan Amount", "Credit Score", "Annual Income", "Monthly Debt",
    "Years of Credit History", "Months since last delinquent",
    "Number of Open Accounts", "Number of Credit Problems",
    "Current Credit Balance", "Maximum Open Credit",
    "Bankruptcies", "Tax Liens",
]

for c in RAW_NUMERIC:
    if c in feat.columns:
        register(c, "raw_numeric", f"Cleaned raw column: {c}", "float", c)

feat_dict = pd.DataFrame(FEATURE_DICT).drop_duplicates(subset=["feature"])
engineered_cols = [c for c in feat_dict["feature"].tolist() if c in feat.columns]

model_cols = []
for c in RAW_NUMERIC + engineered_cols:
    if c in feat.columns and c not in model_cols:
        model_cols.append(c)

drop_helpers = [
    "purpose_group", "credit_score_band", "Purpose", "Term",
    "Home Ownership", "Years in current job", "Loan Status",
]
model_cols = [c for c in model_cols if c not in drop_helpers]

X = feat[model_cols].copy()
y = feat["Default"].copy()
meta = ids.copy()
meta["Default"] = y.values

print(f"Feature matrix X: {X.shape}")
print(f"Target y:         {y.shape}, default rate={y.mean()*100:.2f}%")
print(f"\nFeature list ({len(model_cols)}):")
for i, c in enumerate(model_cols, 1):
    print(f"  {i:2d}. {c}")


## 5. Feature quality checks

Validate missing rates, constant columns, and basic integrity of the engineered matrix.


In [ ]:
miss = (X.isna().mean() * 100).sort_values(ascending=False)
print("Top missing features (%):")
print(miss.head(15).to_string())
print(f"\nFeatures with >0% missing: {(miss > 0).sum()}")
print(f"Features fully populated:  {(miss == 0).sum()}")


In [ ]:
nzv = []
for c in X.columns:
    s = X[c]
    if s.dropna().nunique() <= 1:
        nzv.append((c, "constant"))
    elif s.dropna().nunique() == 2:
        rate = s.mean()
        if rate < 0.005 or rate > 0.995:
            nzv.append((c, f"rare_binary rate={rate:.4f}"))

print(f"Near-zero variance / constant features: {len(nzv)}")
for c, reason in nzv:
    print(f"  - {c}: {reason}")


In [ ]:
assoc = X.apply(lambda col: col.corr(y)).abs().sort_values(ascending=False)
print("Top 20 |corr| with Default:")
print(assoc.head(20).to_string())

fig, ax = plt.subplots(figsize=(9, 7))
top = assoc.head(20)
colors = ["#e74c3c" if X[c].corr(y) > 0 else "#2ecc71" for c in top.index]
ax.barh(top.index[::-1], top.values[::-1], color=colors[::-1])
ax.set_xlabel("|Pearson r| with Default")
ax.set_title("Top 20 features by |correlation| with Default")
plt.tight_layout()
plt.show()


In [ ]:
top_feats = assoc.head(15).index.tolist()
corr_top = X[top_feats].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_top, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=ax, square=True, annot_kws={"size": 7})
ax.set_title("Correlation among top-15 default-associated features")
plt.tight_layout()
plt.show()


In [ ]:
checks = [
    ("is_long_term", "Long Term loan"),
    ("is_renter", "Renter"),
    ("has_delinquency_hist", "Has delinquency history"),
    ("has_any_public_record", "Any public record"),
    ("is_short_tenure", "Short tenure (<=1y)"),
    ("score_band_poor", "Poor credit band"),
    ("longterm_x_renter", "Long-term x Renter"),
    ("purpose_business", "Business purpose"),
]

rows = []
for col, label in checks:
    if col not in X.columns:
        continue
    for val in [0, 1]:
        m = X[col] == val
        if m.sum() == 0:
            continue
        rows.append({
            "feature": label,
            "value": val,
            "n": int(m.sum()),
            "default_rate_%": y[m].mean() * 100,
        })
check_df = pd.DataFrame(rows)
print(check_df.to_string(index=False))


In [ ]:
binary_flags = [
    c for c in X.columns
    if X[c].dropna().nunique() == 2 and X[c].min() in (0, 1) and X[c].max() in (0, 1)
]
lifts = []
for c in binary_flags:
    r0 = y[X[c] == 0].mean()
    r1 = y[X[c] == 1].mean()
    if r0 > 0 and X[c].mean() > 0.01:
        lifts.append({
            "feature": c,
            "rate_0": r0 * 100,
            "rate_1": r1 * 100,
            "lift": r1 / r0,
            "prevalence": X[c].mean() * 100,
        })

lift_df = pd.DataFrame(lifts).sort_values("lift", ascending=False)
print("Binary feature lift (default rate when 1 / when 0):")
print(lift_df.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
top_lift = lift_df.head(12)
ax.barh(top_lift["feature"][::-1], top_lift["lift"][::-1], color="#e67e22")
ax.axvline(1.0, color="black", ls="--", lw=0.8)
ax.set_xlabel("Lift (default rate ratio)")
ax.set_title("Top binary features by default-rate lift")
plt.tight_layout()
plt.show()


## 6. Engineered ratio distributions

Visualise DTI, LTI, utilisation, and related ratios to confirm ranges and outliers after construction.


In [ ]:
ratio_cols = ["dti", "lti", "credit_util", "debt_to_loan", "balance_to_income"]
ratio_cols = [c for c in ratio_cols if c in X.columns]

fig, axes = plt.subplots(1, len(ratio_cols), figsize=(4 * len(ratio_cols), 3.5))
if len(ratio_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, ratio_cols):
    s = X[col].dropna()
    hi = s.quantile(0.99)
    ax.hist(s.clip(upper=hi), bins=40, color="#3498db", edgecolor="white", alpha=0.85)
    ax.set_title(col)
    ax.axvline(s.median(), color="red", ls="--", lw=1, label=f"med={s.median():.2f}")
    ax.legend(fontsize=8)
plt.suptitle("Engineered ratios (clipped at 99th pct for display)", y=1.02)
plt.tight_layout()
plt.show()

X[ratio_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T


## 7. Persist feature set and dictionary

Write parquet/CSV matrices and a human-readable feature dictionary for selection and modelling notebooks.


In [ ]:
out_frame = meta.copy()
out_frame = pd.concat([out_frame.reset_index(drop=True), X.reset_index(drop=True)], axis=1)

parquet_path = OUT_DIR / "credit_train_features.parquet"
csv_path = OUT_DIR / "credit_train_features.csv"
dict_path = OUT_DIR / "feature_dictionary.csv"
cols_path = OUT_DIR / "model_feature_list.json"

out_frame.to_parquet(parquet_path, index=False)
out_frame.to_csv(csv_path, index=False)

feat_dict_final = pd.DataFrame(FEATURE_DICT).drop_duplicates(subset=["feature"])
existing = set(feat_dict_final["feature"])
extra = []
for c in model_cols:
    if c not in existing:
        extra.append({
            "feature": c, "family": "raw_numeric", "description": c,
            "dtype": "float", "source": c,
        })
if extra:
    feat_dict_final = pd.concat([feat_dict_final, pd.DataFrame(extra)], ignore_index=True)
feat_dict_final = feat_dict_final[feat_dict_final["feature"].isin(model_cols)]
feat_dict_final.to_csv(dict_path, index=False)

with open(cols_path, "w") as f:
    json.dump({"features": model_cols, "target": "Default", "id_cols": id_cols}, f, indent=2)

print("Saved:")
print(f"  {parquet_path}  ({parquet_path.stat().st_size/1e6:.1f} MB)")
print(f"  {csv_path}      ({csv_path.stat().st_size/1e6:.1f} MB)")
print(f"  {dict_path}")
print(f"  {cols_path}")
print(f"\nRows: {len(out_frame):,}  |  Feature cols: {len(model_cols)}  |  Target: Default")


## 8. Feature engineering summary

### Families produced

| Family | Role |
|--------|------|
| `raw_numeric` | Cleaned originals (score, income, balances, ...) |
| `ratio` | DTI, LTI, utilisation, debt-to-loan, balance-to-income |
| `transform` | log1p of heavy-tailed amounts |
| `binary_flag` | Term, tenure, home ownership, purpose groups, credit events |
| `missing_flag` | Informative missingness for score, income, delinquency, ... |
| `bin` | Quintile codes for score, income, loan, DTI, util |
| `interaction` | Long-term x renter, long-term x poor score, high-DTI x renter, short-tenure x long-term |
| `ordinal` / `count` | Job years, combined problem count |
| `derived` | Credit headroom |

### Modelling notes

1. **Imputation:** median for ratios/raw numerics; 0 for flags already defined. Prefer adding the corresponding `*_missing` flag rather than dropping rows.
2. **Scaling:** standardise continuous columns for linear models; tree models can use raw values.
3. **Leakage:** no target encoding here. If target encoding is added later, fit only inside CV folds.
4. **Collinearity:** `log_*` transforms and levels are collinear by construction — pick one family per model type (logs for linear, levels or bins for trees).
5. **Rare binaries:** review near-zero variance list before inclusion; `has_tax_lien` may be too sparse for stable estimates.

### Downstream

- Feature matrix: `Kernels/Analytics/features/credit_train_features.parquet`
- Use `model_feature_list.json` as the canonical column order for train/test alignment


In [ ]:
print("=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)
print(f"Observations:     {len(X):,}")
print(f"Features:         {len(model_cols)}")
print(f"Default rate:     {y.mean()*100:.2f}%")
print(f"Top feature:      {assoc.index[0]} (|r|={assoc.iloc[0]:.3f})")
print(f"Output directory: {OUT_DIR.resolve()}")
print("=" * 60)
